# The concept model

A separate, intrinsically text-grounded classifier over the 168 antonym cue pairs. It predicts
the label directly and is *not* fitted to reproduce the linear detector's scores, so read it as
complementary evidence rather than as an explanation of `D_h`.


In [1]:
import pandas as pd
import torch
from PIL import Image
from pyprojroot import here
from transformers import CLIPImageProcessor, CLIPModel

from clip_cues import load_concept_model

ROOT = here()
CACHE = str(ROOT / "data" / "hf_cache")


/workspaces/clip-cues/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load

The checkpoint carries its own concept text embeddings, so the vocabulary comes from the file
rather than being rebuilt — that is what keeps the concept names aligned with the weights.


In [2]:
model, _ = load_concept_model(
    str(ROOT / "data/checkpoints/cm_antonyms_synthclic.ckpt"),
    cache_dir=CACHE,
)
model.eval()

vocab = pd.read_csv(ROOT / "data/vocabularies/antonyms.csv")
print(f"{len(vocab)} concept pairs, e.g. {vocab.attribute_name.iloc[0]!r}")


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 13124.61it/s]
[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14-336
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.final_layer_norm.weight             

168 concept pairs, e.g. 'saturation'


## Features

This model compares images with *text*, so it needs the 768-d shared image–text embedding —
CLIP's `get_image_features` — not the 1024-d pre-projection representation the linear detector
uses. Mixing the two spaces is the most common error in this codebase.


In [18]:
clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14-336", cache_dir=CACHE).eval()
processor = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14-336", cache_dir=CACHE)

image = Image.open(ROOT / "examples/images/synthetic2.jpg").convert("RGB")
with torch.no_grad():
    features = clip.get_image_features(**processor(images=image, return_tensors="pt"))

# transformers >= 5 returns an output object here; earlier versions returned the tensor directly.
embedding = getattr(features, "pooler_output", features)
print("image embedding:", tuple(embedding.shape))


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 12802.46it/s]


image embedding: (1, 768)


## Predict, and see which concepts carried the decision


In [17]:
with torch.no_grad():
    out = model(embedding)

prob = torch.sigmoid(out["class_logits"]).item()
contrib = out["per_concept_logit_contribution"][0].numpy()

print(f"P(synthetic) = {prob:.1%}\n")
top = pd.DataFrame({"concept": vocab.attribute_name, "contribution": contrib})
top = top.reindex(top.contribution.abs().sort_values(ascending=False).index)
print(top.head(10).to_string(index=False))


P(synthetic) = 3.7%

             concept  contribution
      micro_contrast     -0.983396
       bokeh_quality     -0.887436
          retouching      0.840692
sensor_noise_pattern     -0.680430
            symmetry     -0.635281
      scan_artifacts     -0.525893
       leading_lines      0.511990
chromatic_aberration     -0.457896
           sharpness      0.419792
               bloom     -0.410509


## Reading it honestly

Concept *identities* are seed-sensitive: refitting the model reshuffles which named concepts come
out on top, even though the aggregate picture is stable. The claim this model supports is about
the **form** of the explanation — sparse, image-specific, text-grounded — not about any individual
concept being *the* reason.

On SynthCLIC several top concepts fire more on **real** images (`glitch_artifacts`,
`scan_artifacts`, `print_texture`): capture artifacts are evidence of a camera, not of a
generator.
